In [1]:
"""
Grafo COMPLETO K8 — Hospitales de Madrid con Bases de Drones
=============================================================
MEJORAS v3:
  ✔ Coordenadas GPS precisas de cada hospital (entrada principal)
  ✔ Grafo completo K8: 28 aristas, TODAS visibles
  ✔ Cada arista tiene COLOR ÚNICO según los dos hospitales que conecta
  ✔ Nodos (hospitales) mucho más grandes y visibles
  ✔ Etiqueta de km en cada arista
  ✔ 3 bases de drones en ubicaciones REALES y viables de Madrid:
       - BASE NORTE : Parque Dehesa de la Villa (zona arbolada, poco transitada)
       - BASE CENTRO: Parque del Oeste, zona interior (baja densidad)
       - BASE SUR   : Parque Pradolongo, Usera (zona verde tranquila)
  ✔ Justificación de cada base en el popup
  ✔ Radio de cobertura de cada base
  ✔ MST en dorado (activable/desactivable)
  ✔ Exporta a HTML interactivo

Dependencias: pip install folium networkx pandas
"""

import math
import folium
import networkx as nx
import pandas as pd
from itertools import combinations
from folium.plugins import MiniMap

# ─────────────────────────────────────────────────────────────
# 1. HOSPITALES — coordenadas GPS precisas (entrada principal)
# ─────────────────────────────────────────────────────────────
hospitales = {
    "La Paz": {
        "nombre_completo": "H. Universitario La Paz",
        "lat": 40.47330, "lon": -3.68990,   # Paseo de la Castellana 261
        "distrito": "Fuencarral-El Pardo",
        "especialidades": "Neurocirugía, trasplantes, cardíaca, UCI avanzada",
        "color": "#FF6B6B"
    },
    "Gregorio Marañón": {
        "nombre_completo": "H. General Universitario Gregorio Marañón",
        "lat": 40.40950, "lon": -3.68290,   # C/ del Doctor Esquerdo 46
        "distrito": "Retiro",
        "especialidades": "Cirugía compleja, esclerosis múltiple, endoscópica",
        "color": "#4ECDC4"
    },
    "San Carlos": {
        "nombre_completo": "H. Clínico San Carlos",
        "lat": 40.44270, "lon": -3.71860,   # C/ del Prof. Martín Lagos s/n
        "distrito": "Moncloa-Aravaca",
        "especialidades": "Cardiaca, torácica, UCI postoperatorio",
        "color": "#FFE66D"
    },
    "12 de Octubre": {
        "nombre_completo": "H. Universitario 12 de Octubre",
        "lat": 40.38390, "lon": -3.69110,   # Av. de Córdoba s/n
        "distrito": "Usera",
        "especialidades": "Malformaciones vasculares, trasplantes, cirugía mayor",
        "color": "#A8E6CF"
    },
    "Ramón y Cajal": {
        "nombre_completo": "H. Universitario Ramón y Cajal",
        "lat": 40.47240, "lon": -3.64820,   # Ctra. de Colmenar Viejo km 9.1
        "distrito": "Fuencarral-El Pardo",
        "especialidades": "Cirugía esofágica compleja, neurocirugía",
        "color": "#DDA0DD"
    },
    "La Princesa": {
        "nombre_completo": "H. Universitario de La Princesa",
        "lat": 40.42650, "lon": -3.69080,   # C/ de Diego de León 62
        "distrito": "Salamanca",
        "especialidades": "Cirugía general avanzada, UCI",
        "color": "#F7DC6F"
    },
    "Jiménez Díaz": {
        "nombre_completo": "H. Fundación Jiménez Díaz",
        "lat": 40.43530, "lon": -3.72360,   # Av. de los Reyes Católicos 2
        "distrito": "Moncloa-Aravaca",
        "especialidades": "Cirugía torácica de alto riesgo, UCI respiratoria",
        "color": "#85C1E9"
    },
    "Cruz Roja": {
        "nombre_completo": "H. Central de la Cruz Roja",
        "lat": 40.45390, "lon": -3.70540,   # C/ de la Reina Victoria 26
        "distrito": "Tetuán",
        "especialidades": "Cirugía general, emergencias",
        "color": "#F1948A"
    },
}

# ─────────────────────────────────────────────────────────────
# 2. BASES DE DRONES — ubicaciones reales y justificadas
# ─────────────────────────────────────────────────────────────
# Criterios:
#   • Zona verde / área poco edificada (sin riesgo para la población)
#   • Baja densidad de tráfico aéreo comercial
#   • Posición geográfica óptima respecto al cluster de hospitales asignado
#   • Acceso para mantenimiento (vial próximo)

bases_drones = [
    {
        "nombre": "BASE NORTE",
        "lat": 40.46850, "lon": -3.70120,
        "lugar": "Parque Dehesa de la Villa (zona interior norte)",
        "justificacion": (
            "Parque forestal de 64 ha, sendas poco transitadas. "
            "Sin edificios cercanos. Acceso por Av. de Miraflores. "
            "Posición central respecto a La Paz, Ramón y Cajal y Cruz Roja."
        ),
        "hospitales": ["La Paz", "Ramón y Cajal", "Cruz Roja"],
        "color": "#00FF88"
    },
    {
        "nombre": "BASE CENTRO",
        "lat": 40.43820, "lon": -3.71550,
        "lugar": "Parque del Oeste — zona interior (Huerta de la Partida)",
        "justificacion": (
            "Zona arbolada densa del Parque del Oeste, alejada del Templo de Debod. "
            "Baja afluencia peatonal en el interior. Acceso por C/ Fernández de los Ríos. "
            "Posición equidistante entre San Carlos, Jiménez Díaz y La Princesa."
        ),
        "hospitales": ["San Carlos", "Jiménez Díaz", "La Princesa"],
        "color": "#00CFFF"
    },
    {
        "nombre": "BASE SUR",
        "lat": 40.39680, "lon": -3.69850,
        "lugar": "Parque Pradolongo, Usera",
        "justificacion": (
            "Parque urbano con amplias zonas de césped y baja edificación colindante. "
            "Acceso por C/ de Pradillo. Uso moderado. "
            "Posición óptima entre 12 de Octubre y Gregorio Marañón."
        ),
        "hospitales": ["12 de Octubre", "Gregorio Marañón"],
        "color": "#FF9F00"
    },
]

# ─────────────────────────────────────────────────────────────
# 3. HAVERSINE
# ─────────────────────────────────────────────────────────────
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    a = (math.sin((math.radians(lat2 - lat1)) / 2) ** 2
         + math.cos(phi1) * math.cos(phi2)
         * math.sin((math.radians(lon2 - lon1)) / 2) ** 2)
    return round(R * 2 * math.asin(math.sqrt(a)), 3)

nombres = list(hospitales.keys())
distancias = {}
for h1, h2 in combinations(nombres, 2):
    d = haversine_km(hospitales[h1]["lat"], hospitales[h1]["lon"],
                     hospitales[h2]["lat"], hospitales[h2]["lon"])
    distancias[(h1, h2)] = d
    distancias[(h2, h1)] = d

pares_unicos = [(h1, h2, distancias[(h1, h2)]) for h1, h2 in combinations(nombres, 2)]

# ─────────────────────────────────────────────────────────────
# 4. COLOR DE ARISTA: mezcla de colores de los dos hospitales
# ─────────────────────────────────────────────────────────────
def hex_to_rgb(h):
    h = h.lstrip("#")
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))

def mix_colors(c1, c2, alpha=0.5):
    r1, g1, b1 = hex_to_rgb(c1)
    r2, g2, b2 = hex_to_rgb(c2)
    r = int(r1 * alpha + r2 * (1 - alpha))
    g = int(g1 * alpha + g2 * (1 - alpha))
    b = int(b1 * alpha + b2 * (1 - alpha))
    return f"#{r:02x}{g:02x}{b:02x}"

def grosor_arista(d):
    d_min, d_max = 0.5, 11.0
    t = (d - d_min) / (d_max - d_min)
    return round(3.5 - t * 2.0, 1)   # 3.5 (corta) → 1.5 (larga)

# ─────────────────────────────────────────────────────────────
# 5. GRAFO NETWORKX + ESTADÍSTICAS
# ─────────────────────────────────────────────────────────────
G = nx.Graph()
for n in nombres:
    G.add_node(n)
for h1, h2, d in pares_unicos:
    G.add_edge(h1, h2, weight=d)

MST = nx.minimum_spanning_tree(G, weight="weight")
centralidad = nx.betweenness_centrality(G)

dist_vals = [d for _, _, d in pares_unicos]
idx_min   = dist_vals.index(min(dist_vals))
idx_max   = dist_vals.index(max(dist_vals))

print("=" * 64)
print("  GRAFO COMPLETO K8 — HOSPITALES DE MADRID  (v3)")
print("=" * 64)
print(f"  Nodos (hospitales) : {G.number_of_nodes()}")
print(f"  Aristas K8         : {G.number_of_edges()}  (= {len(nombres)}×{len(nombres)-1}/2)")
print(f"  Densidad           : {nx.density(G):.3f}  (1.0 = completo)")
print(f"  Dist. mínima       : {min(dist_vals)} km  "
      f"({pares_unicos[idx_min][0]} ↔ {pares_unicos[idx_min][1]})")
print(f"  Dist. máxima       : {max(dist_vals)} km  "
      f"({pares_unicos[idx_max][0]} ↔ {pares_unicos[idx_max][1]})")
print(f"  Dist. media        : {sum(dist_vals)/len(dist_vals):.2f} km")
print(f"  Peso total MST     : {sum(MST[u][v]['weight'] for u,v in MST.edges()):.2f} km")
print()
print("  CENTRALIDAD (mayor = más estratégico):")
for h, c in sorted(centralidad.items(), key=lambda x: -x[1]):
    print(f"    {h:<22} {c:.4f}")
print()
print("  TABLA DE DISTANCIAS AÉREAS (km):")
df = pd.DataFrame(0.0, index=nombres, columns=nombres)
for h1, h2, d in pares_unicos:
    df.at[h1, h2] = d
    df.at[h2, h1] = d
print(df.round(2).to_string())
print()
print("  BASES DE DRONES:")
for b in bases_drones:
    print(f"\n  🚁 {b['nombre']}  →  {b['lat']}, {b['lon']}")
    print(f"     Ubicación : {b['lugar']}")
    for h in b["hospitales"]:
        d = haversine_km(hospitales[h]["lat"], hospitales[h]["lon"], b["lat"], b["lon"])
        print(f"       • {h:<25}  {d:.2f} km")
print("=" * 64)

# ─────────────────────────────────────────────────────────────
# 6. MAPA INTERACTIVO FOLIUM
# ─────────────────────────────────────────────────────────────
mapa = folium.Map(
    location=[40.4300, -3.6950],
    zoom_start=12,
    tiles="CartoDB dark_matter",
    control_scale=True
)
MiniMap(toggle_display=True).add_to(mapa)

# ── CAPA 1: GRAFO COMPLETO K8 ─────────────────────────────────
fg_completo = folium.FeatureGroup(name="🔗 Grafo completo K8 (28 aristas)", show=True)

for h1, h2, dist in pares_unicos:
    p1 = [hospitales[h1]["lat"], hospitales[h1]["lon"]]
    p2 = [hospitales[h2]["lat"], hospitales[h2]["lon"]]

    # Color = mezcla de los colores de los dos hospitales
    color  = mix_colors(hospitales[h1]["color"], hospitales[h2]["color"])
    grosor = grosor_arista(dist)
    mid    = [(p1[0] + p2[0]) / 2, (p1[1] + p2[1]) / 2]

    folium.PolyLine(
        locations=[p1, p2],
        color=color,
        weight=grosor,
        opacity=0.85,
        tooltip=f"✈️ {h1} ↔ {h2}  →  {dist} km"
    ).add_to(fg_completo)

    # Etiqueta km en punto medio
    folium.Marker(
        location=mid,
        icon=folium.DivIcon(
            html=(
                f'<div style="font-size:8px; font-weight:bold; color:{color}; '
                f'background:rgba(0,0,0,0.72); padding:1px 4px; border-radius:3px; '
                f'white-space:nowrap; border:1px solid {color}66;">'
                f'{dist} km</div>'
            ),
            icon_size=(56, 16),
            icon_anchor=(28, 8)
        )
    ).add_to(fg_completo)

fg_completo.add_to(mapa)

# ── CAPA 2: MST (desactivado por defecto para no saturar) ─────
fg_mst = folium.FeatureGroup(name="⭐ Red MST — coste mínimo", show=False)
for h1, h2 in MST.edges():
    dist = distancias[(h1, h2)]
    p1   = [hospitales[h1]["lat"], hospitales[h1]["lon"]]
    p2   = [hospitales[h2]["lat"], hospitales[h2]["lon"]]
    mid  = [(p1[0] + p2[0]) / 2, (p1[1] + p2[1]) / 2]
    folium.PolyLine(
        locations=[p1, p2],
        color="#FFD700", weight=5, opacity=1.0,
        tooltip=f"⭐ MST: {h1} ↔ {h2}  →  {dist} km",
        dash_array="12"
    ).add_to(fg_mst)
    folium.Marker(
        location=mid,
        icon=folium.DivIcon(
            html=(f'<div style="font-size:9px; font-weight:bold; color:#FFD700; '
                  f'background:rgba(0,0,0,0.82); padding:2px 5px; border-radius:4px; '
                  f'border:1px solid #FFD700; white-space:nowrap;">{dist} km</div>'),
            icon_size=(60, 18), icon_anchor=(30, 9)
        )
    ).add_to(fg_mst)
fg_mst.add_to(mapa)

# ── CAPA 3: HOSPITALES — nodos grandes y bien centrados ───────
fg_hospitales = folium.FeatureGroup(name="🏥 Hospitales", show=True)
for nombre, datos in hospitales.items():
    c     = centralidad[nombre]
    radio = 18 + c * 100   # mucho más grande: 18–28 px aprox

    filas = "".join(
        f'<tr><td style="padding:2px 8px;color:#bbb;">↔ {h2}</td>'
        f'<td style="padding:2px 8px;font-weight:bold;color:white;">'
        f'{distancias[(nombre, h2)]} km</td></tr>'
        for h2 in nombres if h2 != nombre
    )
    popup_html = f"""
    <div style="font-family:'Segoe UI',sans-serif;min-width:280px;max-width:330px;
                background:#111;color:white;padding:6px;">
        <h4 style="margin:0 0 8px;color:{datos['color']};
                   border-bottom:2px solid {datos['color']};padding-bottom:4px;">
            🏥 {datos['nombre_completo']}
        </h4>
        <p style="margin:3px 0;"><b>📍 Distrito:</b> {datos['distrito']}</p>
        <p style="margin:3px 0;"><b>🔬</b> {datos['especialidades']}</p>
        <p style="margin:3px 0;"><b>📡 Centralidad:</b> {c:.4f}</p>
        <p style="margin:3px 0;"><b>🗺️</b> {datos['lat']:.5f}, {datos['lon']:.5f}</p>
        <hr style="border-color:#333;margin:6px 0;">
        <p style="margin:3px 0;font-size:11px;color:#aaa;">Distancias aéreas a todos:</p>
        <table style="font-size:11px;width:100%;">{filas}</table>
    </div>
    """

    # Círculo exterior (halo) para mayor visibilidad
    folium.CircleMarker(
        location=[datos["lat"], datos["lon"]],
        radius=radio + 4,
        color=datos["color"],
        fill=False,
        weight=1.5,
        opacity=0.4
    ).add_to(fg_hospitales)

    # Nodo principal
    folium.CircleMarker(
        location=[datos["lat"], datos["lon"]],
        radius=radio,
        color=datos["color"],
        fill=True,
        fill_color=datos["color"],
        fill_opacity=0.90,
        weight=2.5,
        popup=folium.Popup(popup_html, max_width=340),
        tooltip=f"🏥 {nombre}  |  Centralidad: {c:.3f}"
    ).add_to(fg_hospitales)

    # Etiqueta de nombre
    folium.Marker(
        location=[datos["lat"] + 0.0032, datos["lon"]],
        icon=folium.DivIcon(
            html=(f'<div style="font-size:11px;font-weight:bold;color:{datos["color"]};'
                  f'text-shadow:1px 1px 4px #000,-1px -1px 4px #000;white-space:nowrap;">'
                  f'{nombre}</div>'),
            icon_size=(180, 22),
            icon_anchor=(90, 11)
        )
    ).add_to(fg_hospitales)

fg_hospitales.add_to(mapa)

# ── CAPA 4: BASES DE DRONES ────────────────────────────────────
fg_bases = folium.FeatureGroup(name="🚁 Bases de drones (ubicaciones reales)", show=True)
for base in bases_drones:
    # Radio de cobertura = distancia al hospital más lejano asignado
    max_d = max(
        haversine_km(hospitales[h]["lat"], hospitales[h]["lon"],
                     base["lat"], base["lon"])
        for h in base["hospitales"]
    )

    # Círculo de cobertura
    folium.Circle(
        location=[base["lat"], base["lon"]],
        radius=max_d * 1000,
        color=base["color"],
        fill=True, fill_opacity=0.06,
        weight=2, dash_array="7",
        tooltip=f"Cobertura {base['nombre']}: {max_d:.2f} km radio"
    ).add_to(fg_bases)

    # Líneas base → hospitales asignados
    for h in base["hospitales"]:
        d = haversine_km(hospitales[h]["lat"], hospitales[h]["lon"],
                         base["lat"], base["lon"])
        folium.PolyLine(
            locations=[[base["lat"], base["lon"]],
                        [hospitales[h]["lat"], hospitales[h]["lon"]]],
            color=base["color"],
            weight=2.2, opacity=0.80,
            dash_array="6",
            tooltip=f"🚁 {base['nombre']} → {h}: {d:.2f} km"
        ).add_to(fg_bases)

    # Popup base con justificación
    filas_b = "".join(
        f'<tr><td style="padding:2px 6px;color:#ccc;">• {h}</td>'
        f'<td style="padding:2px 6px;font-weight:bold;color:white;">'
        f'{haversine_km(hospitales[h]["lat"],hospitales[h]["lon"],base["lat"],base["lon"]):.2f} km'
        f'</td></tr>'
        for h in base["hospitales"]
    )
    popup_b = f"""
    <div style="font-family:'Segoe UI',sans-serif;min-width:260px;max-width:320px;
                background:#111;color:white;padding:6px;">
        <h4 style="color:{base['color']};margin:0 0 8px;
                   border-bottom:2px solid {base['color']};padding-bottom:4px;">
            🚁 {base['nombre']}
        </h4>
        <p style="margin:3px 0;"><b>📍 Ubicación real:</b><br>
           <span style="color:#aaa;font-size:11px;">{base['lugar']}</span></p>
        <p style="margin:6px 0;"><b>✅ Por qué aquí:</b><br>
           <span style="color:#aaa;font-size:11px;">{base['justificacion']}</span></p>
        <p style="margin:3px 0;"><b>🗺️ Coords:</b> {base['lat']}, {base['lon']}</p>
        <p style="margin:3px 0;"><b>📏 Radio cobertura:</b> {max_d:.2f} km</p>
        <hr style="border-color:#333;margin:6px 0;">
        <table style="font-size:11px;width:100%;">{filas_b}</table>
    </div>
    """
    folium.Marker(
        location=[base["lat"], base["lon"]],
        popup=folium.Popup(popup_b, max_width=330),
        tooltip=f"🚁 {base['nombre']} — {base['lugar']}",
        icon=folium.DivIcon(
            html=(f'<div style="font-size:28px;text-align:center;'
                  f'filter:drop-shadow(0 0 10px {base["color"]});">🚁</div>'),
            icon_size=(36, 36), icon_anchor=(18, 18)
        )
    ).add_to(fg_bases)

fg_bases.add_to(mapa)

# ── LEYENDA ────────────────────────────────────────────────────
leyenda_html = """
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;
     background:rgba(8,8,18,0.95);border:1px solid #FFD700;
     border-radius:10px;padding:15px 19px;
     font-family:'Segoe UI',sans-serif;color:white;font-size:12px;
     min-width:250px;box-shadow:0 4px 24px rgba(0,0,0,0.7);">
  <b style="color:#FFD700;font-size:14px;">🗺️ Grafo K8 — Hospitales Madrid</b>
  <hr style="border-color:#333;margin:8px 0;">
  <b style="color:#aaa;">Aristas:</b> color = mezcla de los 2 hospitales<br>
  &nbsp;&nbsp;&nbsp;Grosor mayor → distancia más corta<br>
  <span style="color:#FFD700;">╌╌╌╌</span> MST (activar en capas)<br>
  <hr style="border-color:#333;margin:8px 0;">
  <b style="color:#aaa;">Bases de drones:</b><br>
  <span style="color:#00FF88;">🚁</span> BASE NORTE — Dehesa de la Villa<br>
  <span style="color:#00CFFF;">🚁</span> BASE CENTRO — Parque del Oeste<br>
  <span style="color:#FF9F00;">🚁</span> BASE SUR — Parque Pradolongo<br>
  <hr style="border-color:#333;margin:8px 0;">
  <i style="color:#888;font-size:10px;">Clic en hospitales/bases → detalles<br>
  Panel lateral → activar/desactivar capas</i>
</div>
"""
mapa.get_root().html.add_child(folium.Element(leyenda_html))

folium.LayerControl(collapsed=False).add_to(mapa)

# ── GUARDAR ───────────────────────────────────────────────────
output = "hospitales_madrid_grafo.html"
mapa.save(output)
print(f"\n✅ Mapa guardado en: {output}")
print("   Ábrelo en tu navegador para explorar el grafo completo K8.")

  GRAFO COMPLETO K8 — HOSPITALES DE MADRID  (v3)
  Nodos (hospitales) : 8
  Aristas K8         : 28  (= 8×7/2)
  Densidad           : 1.000  (1.0 = completo)
  Dist. mínima       : 0.925 km  (San Carlos ↔ Jiménez Díaz)
  Dist. máxima       : 10.489 km  (12 de Octubre ↔ Ramón y Cajal)
  Dist. media        : 5.05 km
  Peso total MST     : 16.53 km

  CENTRALIDAD (mayor = más estratégico):
    La Paz                 0.0000
    Gregorio Marañón       0.0000
    San Carlos             0.0000
    12 de Octubre          0.0000
    Ramón y Cajal          0.0000
    La Princesa            0.0000
    Jiménez Díaz           0.0000
    Cruz Roja              0.0000

  TABLA DE DISTANCIAS AÉREAS (km):
                  La Paz  Gregorio Marañón  San Carlos  12 de Octubre  Ramón y Cajal  La Princesa  Jiménez Díaz  Cruz Roja
La Paz              0.00              7.12        4.18           9.94           3.53         5.20          5.10       2.52
Gregorio Marañón    7.12              0.00        4.77  